# Run analysis: `run-10.1_multi-day-rawfeat+featdeviation+labelhist-max-p5fcv`

Test-period analysis of a rolling-window training run. Compared against the baseline run `run-5_baseline-all-feat-purged-5-fold-val-retrain`.

Reads the run's `results_lgbm.json`, `predictions/` and `curves/`. Saves every
figure to `plots/analysis_*.png`. Generated by the `analyze-ts48-results`
skill; re-run top to bottom.

Sections:
1. Daily metrics: Pearson IC, Rank IC (Spearman), daily Kendall, Q5−Q1
2. Per-window summary: ICIR, Newey–West t, train/val/test R², cost
3. Plots: IC over time, cumulative IC, per-window metrics, R², quintiles,
   calibration
4. Finer grain: months and robust summaries
5. Ex-ante regimes
6. What changed vs the baseline: setup differences, automatic script diff,
   row and feature counts (only if a baseline is set)
7. Comparison with the baseline: paired daily differences on the same stocks
   and dates (only if a baseline is set)
8. Findings, summary and hypotheses (hypotheses only with a baseline)

**Ground rules** (from the baseline analysis, `claude/project-notes.md`):
- **Ranking and magnitude both matter.** The predictions feed a portfolio
  optimizer. Rank IC is the robust ranking measure (Pearson IC is inflated
  by fat-tailed labels); calibration slope, R² and RMSE judge magnitudes.
- **The label** is roughly a 5-day forward residual return. No further
  financial meaning is attached; values are in label units.
- **Significance uses Newey–West t.** Overlapping 5-day labels make daily
  ICs autocorrelated, so naive t-stats are ~1.7× too high.
- **Months are diagnostics.** ~20 days is only ~5 independent days, and
  months are never excluded from headline numbers.
- **Regimes use only information known at T.**
- **Comparisons are paired** on the same (Date, Code) rows.
- **The final reserved window** is evaluated only once, at the end of the
  project.


In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.patches import Patch
from scipy import stats

# The notebook lives in the run folder; VS Code runs it from there.
RUN_DIR = Path.cwd()
BASELINE_DIR = Path("/workspaces/TimeSeries48/runs/run-5_baseline-all-feat-purged-5-fold-val-retrain")  # Path to the baseline run folder, or None (set by the skill).
assert (RUN_DIR / "results_lgbm.json").exists(), f"Not a run folder: {RUN_DIR}"
COMPARE = BASELINE_DIR is not None and BASELINE_DIR.resolve() != RUN_DIR.resolve()
DATA_FILE = RUN_DIR.parents[1] / "data.parquet"
PLOTS_DIR = RUN_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Fixed categorical order, never cycled: one colour per window (7th = final window).
WINDOW_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
SPLIT_COLORS = {"train": "#52514e", "val": "#6250d6", "test": "#e34948"}
MODEL_COLORS = {"baseline": "#8f8e88", "run": "#2a78d6"}
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2,
    "legend.frameon": False,
})
warnings.filterwarnings("ignore", message=".*non-interactive.*")


def save(fig, name: str) -> None:
    """Save a figure to plots/analysis_<name>.png, then show it."""
    fig.savefig(PLOTS_DIR / f"analysis_{name}.png", dpi=120, bbox_inches="tight")
    plt.show()


def newey_west_t(x: np.ndarray, lags: int = 4) -> float:
    """t-statistic of mean(x) with a Newey–West (Bartlett-weighted) standard error."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    e = x - x.mean()
    n = len(x)
    var = e @ e / n
    for lag in range(1, min(lags, n - 1) + 1):
        var += 2 * (1 - lag / (lags + 1)) * (e[lag:] @ e[:-lag]) / n
    return float(x.mean() / np.sqrt(var / n))


def nw_ci(x: np.ndarray) -> float:
    """Half-width of a 95% confidence interval for mean(x), Newey–West (4 lags)."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    t = newey_west_t(x)
    return float(1.96 * abs(x.mean() / t)) if t else float("nan")


def find_files(run_dir: Path, kind: str, pattern: str) -> list[Path]:
    """Files in run_dir/<kind>/ (current layout) or run_dir/ (early runs' flat layout)."""
    sub = run_dir / kind
    files = sorted(sub.glob(pattern)) if sub.is_dir() else []
    return files or sorted(run_dir.glob(pattern))


def load_run(run_dir: Path) -> tuple[list[dict], pl.DataFrame, dict[int, dict]]:
    """Results, test predictions (with a 1-based `window` column) and learning curves."""
    results = json.loads((run_dir / "results_lgbm.json").read_text())
    start_to_window = {r["test_start"]: k + 1 for k, r in enumerate(results)}

    def window_of(path: Path) -> int:
        return start_to_window[re.search(r"_test_(\d{4}-\d{2}-\d{2})_", path.name)[1]]

    preds = pl.concat([
        pl.read_parquet(f).with_columns(pl.lit(window_of(f)).cast(pl.Int32).alias("window"))
        for f in find_files(run_dir, "predictions", "predictions_*.parquet")
    ]).sort(["window", "Date", "Code"])
    curves = {window_of(f): json.loads(f.read_text())
              for f in find_files(run_dir, "curves", "curves_*.json")}
    return results, preds, curves


def daily_metrics(preds: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Per-date IC, Rank IC, Kendall and Q5−Q1; plus quintile means and preds with quintiles."""
    daily = (
        preds.group_by(["window", "Date"], maintain_order=True)
        .agg(
            pl.corr("Prediction", "label").alias("ic"),
            pl.corr("Prediction", "label", method="spearman").alias("rank_ic"),
            pl.len().alias("n"),
        )
        .sort(["window", "Date"])
    )
    kendall = [
        stats.kendalltau(d["Prediction"].to_numpy(), d["label"].to_numpy()).statistic
        for d in preds.sort(["window", "Date"]).partition_by(["window", "Date"], maintain_order=True)
    ]
    daily = daily.with_columns(pl.Series("kendall", kendall))
    key = ["window", "Date"]
    preds = preds.with_columns(
        ((pl.col("Prediction").rank("ordinal").over(key).cast(pl.Int64) - 1) * 5
         // pl.len().over(key) + 1).alias("quintile")
    )
    quint_daily = preds.group_by(["window", "Date", "quintile"]).agg(
        pl.col("label").mean().alias("label_mean")
    )
    spread = (
        quint_daily.pivot(on="quintile", index=["window", "Date"], values="label_mean")
        .select("window", "Date", (pl.col("5") - pl.col("1")).alias("q5_minus_q1"))
    )
    daily = daily.join(spread, on=["window", "Date"]).sort(["window", "Date"])
    return daily, quint_daily, preds


In [ ]:
results, preds, curves = load_run(RUN_DIR)
n_windows = len(results)
assert preds["window"].n_unique() == n_windows, "predictions missing for some windows"
print(f"RUN {RUN_DIR.name}: {n_windows} windows, {preds.height:,} test rows, "
      f"{preds['Date'].n_unique()} test dates ({preds['Date'].min()} -> {preds['Date'].max()})")
if len(curves) != n_windows:
    print(f"  note: learning curves found for {len(curves)} of {n_windows} windows")

if COMPARE:
    b_results, b_preds, b_curves = load_run(BASELINE_DIR)
    print(f"BASELINE {BASELINE_DIR.name}: {len(b_results)} windows, {b_preds.height:,} rows, "
          f"{b_preds['Date'].min()} -> {b_preds['Date'].max()}")
    run_windows = {(r["test_start"], r["test_end"]) for r in results}
    base_windows = {(r["test_start"], r["test_end"]) for r in b_results}
    if run_windows != base_windows:
        print("  WARNING: test windows differ between run and baseline; "
              "the comparison uses only their common (Date, Code) rows.")


## 1. Daily metrics

Every metric is computed **within each test date** (stocks vs stocks on the
same day), then summarised per window.

- **IC:** Pearson correlation of prediction and label. Uses magnitudes, so
  it is sensitive to extreme stocks.
- **Rank IC:** Spearman correlation, i.e. ordering only. The robust ranking
  metric (magnitudes are judged separately, by calibration and R²).
- **Kendall:** pairwise ordering agreement; numerically ≈ 0.68 × Rank IC.
- **Q5−Q1:** mean label of the top prediction quintile minus the bottom
  quintile, in label units (the label is roughly a 5-day forward residual
  return; no further financial meaning is attached). Gross, not a profit.


In [ ]:
daily, quint_daily, preds = daily_metrics(preds)
daily.head()


## 2. Per-window summary

- **ICIR** = mean(IC) / std(IC), unscaled, so it is comparable across windows.
  The training script's `ir` multiplies by √days and is really a naive
  t-statistic.
- **t (naive)** assumes independent days. **t (NW)** uses Newey–West standard
  errors with 4 lags, needed because the 5-day labels overlap. **IC ac1** is
  the lag-1 autocorrelation of daily IC.
- **train / val R²** at `best_iteration`, from the learning curves; **test
  R²** from the results. val ≫ test can mean the interleaved-validation leak
  (I-1) **or** a hard test period: check months (section 4) before blaming
  the leak.
- R² and calibration are meaningful only if predictions are in label units.
  They are not if the run trains on transformed labels (e.g. rank-Gauss).


In [ ]:
def curve_at(c: dict | None, split: str, metric: str) -> float:
    """A learning-curve metric at best_iteration; NaN if it wasn't recorded."""
    if not c or not c.get("best_iteration"):
        return float("nan")
    values = c["curves"].get(split, {}).get(metric)
    best = c["best_iteration"]
    return values[best - 1] if values and best <= len(values) else float("nan")


rows = []
for w in range(1, n_windows + 1):
    d = daily.filter(pl.col("window") == w)
    ic = d["ic"].drop_nans().to_numpy()
    r = results[w - 1]
    rows.append({
        "window": w,
        "test": f"{r['test_start']} -> {r['test_end']}",
        "days": len(ic),
        "features": r.get("features"),
        "best_iter": r.get("best_iteration"),
        "IC": ic.mean(),
        "RankIC": d["rank_ic"].mean(),
        "Kendall(daily)": d["kendall"].mean(),
        "ICIR": ic.mean() / ic.std(ddof=1),
        "IC>0 %": 100 * (ic > 0).mean(),
        "t(naive)": ic.mean() / ic.std(ddof=1) * np.sqrt(len(ic)),
        "t(NW)": newey_west_t(ic),
        "IC ac1": np.corrcoef(ic[1:], ic[:-1])[0, 1],
        "train R2": curve_at(curves.get(w), "train", "r2"),
        "val R2": curve_at(curves.get(w), "val", "r2"),
        "test R2": r.get("r2", float("nan")),
        "Q5-Q1": d["q5_minus_q1"].mean(),
        "runtime s": r.get("runtime", float("nan")),
        "peak GB": r.get("peak_memory_GB", r.get("peak_memory_gb", float("nan"))),
    })
summary = pl.DataFrame(rows)
all_ic = daily["ic"].drop_nans().to_numpy()
all_ric = daily["rank_ic"].drop_nans().to_numpy()
print(f"All windows: RankIC={all_ric.mean():.4f} (t NW {newey_west_t(all_ric):.1f})  "
      f"IC={all_ic.mean():.4f}  ICIR={all_ic.mean() / all_ic.std(ddof=1):.2f}  "
      f"t(NW)={newey_west_t(all_ic):.1f}  IC>0 on {100 * (all_ic > 0).mean():.0f}% of days")
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    display(summary)  # noqa: F821  (IPython built-in)


## 3. Plots

- **IC over time:** daily values (grey) and a 20-day rolling mean per window
  (colour).
- **Cumulative IC:** the slope is the IC level. Flat stretches are periods
  where the signal stopped working. Not a profit curve.
- **Per-window metrics and R²:** train / val / test R² side by side.
- **Quintiles:** mean label per prediction quintile. In the baseline the edge
  sits in Q1: volatile, lottery-like stocks drift down, while big winners are
  news-driven and unpredictable.
- **Calibration:** 20 prediction bins per window, (mean prediction, mean
  label). On y = x means calibrated; flatter means overconfident.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True, sharey=True)
for ax, col, name in zip(axes, ["ic", "rank_ic"], ["Pearson IC", "Rank IC (Spearman)"]):
    for w in range(1, n_windows + 1):
        d = daily.filter(pl.col("window") == w)
        x = d["Date"].to_numpy()
        ax.plot(x, d[col], color="#b5b4ad", lw=0.7)
        ax.plot(x, d[col].rolling_mean(20, min_samples=5),
                color=WINDOW_COLORS[w - 1], label=f"W{w} (20-day mean)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Daily {name}: grey = daily, colour = 20-day rolling mean per window")
axes[0].legend(ncol=n_windows, loc="upper left", fontsize=8)
save(fig, "ic_over_time")

fig, ax = plt.subplots(figsize=(14, 3.8))
x = daily["Date"].to_numpy()
ax.plot(x, daily["ic"].fill_nan(0).cum_sum(), color=WINDOW_COLORS[0], label="Pearson IC")
ax.plot(x, daily["rank_ic"].fill_nan(0).cum_sum(), color=WINDOW_COLORS[1], label="Rank IC")
for w in range(2, n_windows + 1):
    ax.axvline(daily.filter(pl.col("window") == w)["Date"][0], color="gray", ls="--", lw=0.8)
ax.set_title("Cumulative daily IC (slope = IC level; not a profit curve). Dashed: new window/model")
ax.legend()
save(fig, "cumulative_ic")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
wins = summary["window"].to_list()
for ax, col in zip(axes, ["IC", "RankIC", "Kendall(daily)"]):
    ax.bar(wins, summary[col], color=WINDOW_COLORS[:n_windows], width=0.7)
    ax.set_title(f"mean daily {col.replace('(daily)', '')}")
    ax.set_xticks(wins, [f"W{w}" for w in wins])
save(fig, "metrics_by_window")

fig, ax = plt.subplots(figsize=(8, 3.8))
for split, col in [("train", "train R2"), ("val", "val R2"), ("test", "test R2")]:
    ax.plot(wins, summary[col], "o-", color=SPLIT_COLORS[split], label=split, ms=8)
ax.set_xticks(wins, [f"W{w}" for w in wins])
ax.set_title("R² by window (train/val at best_iteration)")
ax.legend()
save(fig, "r2_by_window")

quint = (
    quint_daily.group_by(["window", "quintile"]).agg(pl.col("label_mean").mean())
    .sort(["window", "quintile"])
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for w in range(1, n_windows + 1):
    q = quint.filter(pl.col("window") == w)
    axes[0].plot(q["quintile"], q["label_mean"], "o-", color=WINDOW_COLORS[w - 1], label=f"W{w}", ms=6)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].set_xlabel("prediction quintile (1 = lowest predicted)")
axes[0].set_title("Mean label by prediction quintile")
axes[0].legend(ncol=2)
binned = (
    preds.with_columns(
        ((pl.col("Prediction").rank("ordinal").over("window").cast(pl.Int64) - 1) * 20
         // pl.len().over("window")).alias("bin")
    )
    .group_by(["window", "bin"]).agg(pl.col("Prediction").mean(), pl.col("label").mean())
    .sort(["window", "bin"])
)
for w in range(1, n_windows + 1):
    b = binned.filter(pl.col("window") == w)
    axes[1].plot(b["Prediction"], b["label"], "o-", color=WINDOW_COLORS[w - 1], label=f"W{w}", ms=4, lw=1.5)
lim = [binned["Prediction"].min(), binned["Prediction"].max()]
axes[1].plot(lim, lim, "--", color="gray", lw=1, label="y = x")
axes[1].set_xlabel("mean prediction in bin")
axes[1].set_ylabel("mean label in bin")
axes[1].set_title("Calibration (20 prediction bins per window)")
axes[1].legend(ncol=2, fontsize=8)
save(fig, "quintiles_calibration")
print(quint.pivot(on="quintile", index="window", values="label_mean"))


## 4. Finer grain: months and robust summaries

A weak window can mean broad weakness, or a few bad stretches inside an
otherwise good period. This section separates the two.

**How to read it.** A single month (~20 test days, with strongly
autocorrelated daily ICs) is worth only ~5 independent days. One bad month
proves little; look for **patterns**: months that are bad across several
models, or runs of weak months. Headline metrics stay full-period. Months are
never excluded after the fact (that would be cherry-picking).

- **Top panel:** monthly mean daily **Rank IC** (bars, colour = window) and
  **Pearson IC** (black dots). Dots far above their bar mark months where the
  signal worked only through a few extreme stocks.
- **Bottom panel:** monthly **calibration slope**, the regression slope of
  label on prediction. 1 means magnitudes are right; below 1 means
  overconfident (predictions too spread out); below 0.5 (dotted) means
  strongly overconfident.
- **Robust table:** median daily IC and a 10%-trimmed mean. If they are close
  to the mean, the window's result isn't driven by a few extreme days.


In [ ]:
def month_stats(frame: pl.DataFrame, by: list[str]) -> pl.DataFrame:
    """Pooled R² and calibration slope per group of rows."""
    return frame.group_by(by).agg(
        (1 - ((pl.col("label") - pl.col("Prediction")) ** 2).sum()
         / ((pl.col("label") - pl.col("label").mean()) ** 2).sum()).alias("R2"),
        (pl.cov("Prediction", "label") / pl.col("Prediction").var()).alias("slope"),
    )


preds = preds.with_columns(pl.col("Date").dt.strftime("%Y-%m").alias("month"))
daily = daily.with_columns(pl.col("Date").dt.strftime("%Y-%m").alias("month"))
monthly = (
    daily.group_by(["window", "month"]).agg(
        pl.len().alias("days"),
        pl.col("ic").mean().alias("IC"),
        pl.col("rank_ic").mean().alias("RankIC"),
        (100 * (pl.col("ic") > 0).mean()).alias("IC>0 %"),
    )
    .join(month_stats(preds, ["window", "month"]), on=["window", "month"])
    .sort(["month", "window"])
)
with pl.Config(tbl_rows=-1, float_precision=3):
    display(monthly)  # noqa: F821

x = np.arange(monthly.height)
colors = [WINDOW_COLORS[w - 1] for w in monthly["window"]]
labels_x = [m if d >= 5 else f"{m}*" for m, d in zip(monthly["month"], monthly["days"])]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].bar(x, monthly["RankIC"], color=colors, width=0.8)
axes[0].plot(x, monthly["IC"], "o", color="#0b0b0b", ms=4, label="Pearson IC")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].axhline(daily["rank_ic"].mean(), color="gray", ls="--", lw=1,
                label=f"overall mean Rank IC ({daily['rank_ic'].mean():.3f})")
axes[0].set_ylabel("correlation")
axes[0].set_title("Monthly mean daily Rank IC (bars, colour = window) and Pearson IC (dots)", pad=20)
axes[0].legend(loc="upper left", fontsize=8)
axes[1].bar(x, monthly["slope"], color=colors, width=0.8)
axes[1].axhline(1, color="gray", ls="--", lw=1, label="calibrated (slope 1)")
axes[1].axhline(0.5, color="gray", ls=":", lw=1, label="strongly overconfident (< 0.5)")
axes[1].set_ylabel("slope of label on prediction")
axes[1].set_title("Monthly calibration slope (< 1 = predictions too spread out)")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_xticks(x, labels_x, rotation=60, fontsize=8)
# Window labels above the top panel.
for w in range(1, n_windows + 1):
    idx = np.flatnonzero(monthly["window"].to_numpy() == w)
    axes[0].annotate(f"W{w}", (idx.mean(), 1.0), xycoords=("data", "axes fraction"),
                     ha="center", va="bottom", fontsize=9, color=WINDOW_COLORS[w - 1])
fig.text(0.01, 0.0, "* fewer than 5 test days in that month for that window", fontsize=8)
save(fig, "monthly")

robust = daily.group_by("window").agg(
    pl.col("ic").mean().alias("IC mean"),
    pl.col("ic").median().alias("IC median"),
    pl.col("ic").filter(
        pl.col("ic").is_between(pl.col("ic").quantile(0.1), pl.col("ic").quantile(0.9))
    ).mean().alias("IC trimmed 10%"),
    pl.col("rank_ic").median().alias("RankIC median"),
    (100 * (pl.col("rank_ic") > 0).mean()).alias("RankIC>0 %"),
).sort("window")
with pl.Config(float_precision=4):
    display(robust)  # noqa: F821


## 5. Ex-ante regimes: does performance depend on observable conditions?

**What a regime split is.** Each test day is put into a group ("regime")
according to the market conditions on that day, and performance is compared
across groups.

**Why ex-ante (known in advance).** A regime split is only useful if you could
have *acted* on it: trust or size the signal more in some conditions, add the
condition as a model feature, or check whether a model variant (e.g. history
features) helps mainly in certain conditions. So each day's regime is built
**only from information available at the close of day T**. Labelling months
after the fact ("that was a crash month") can explain the past, but nobody
could have traded on it, and it invites cherry-picking. A regime split does
**not** try to foresee shocks. Finding that the regime *can't* flag the bad
periods is itself a useful (negative) result.

**How the regime is built.**
1. *Dispersion* = the cross-sectional std of the label on a date: how
   differently stocks moved. Labels are factor-adjusted, so this is
   stock-specific dispersion, **not** market volatility.
2. *Trailing dispersion* at T = the mean dispersion over dates T−26 … T−7.
   Those labels are fully realized by day T, so there is no look-ahead.
3. *Percentile* = where today's trailing dispersion ranks among the previous
   ~250 trading days (one year, at least 120 needed). Below the 33rd
   percentile is **calm**, above the 67th is **turbulent**, and in between is
   **normal**. Relative to the past year, so the thresholds adapt over time.

**How to read the charts.**
- *Timeline* (three rows sharing the date axis):
  1. Daily dispersion (thin) and the trailing measure (bold). Dotted lines
     mark the largest dispersion spikes. Note how the trailing measure reacts
     only after a spike: shock onsets look calm ex-ante.
  2. The percentile and the 1/3 and 2/3 thresholds. Background shading =
     regime (light = calm, dark = turbulent), the same in all rows.
  3. Daily Rank IC (faint) and its 20-day mean (bold), against the regime
     shading.
- *Summary*: left, mean daily Rank IC per regime, with Newey–West 95%
  confidence intervals and the number of days and of distinct **episodes**
  (separate runs of ≥ 5 days). Few episodes means weak evidence, however many
  days they cover. Right, the same per window: if the effect is real it
  should appear in most windows, not just one.

**In the comparison (section 6)** the same regimes are used to check *where*
a variant helps: e.g. whether history features mainly help after dispersion
spikes.


In [ ]:
LAG, SPAN, HISTORY, MIN_HISTORY = 7, 20, 250, 120
per_day = (
    pl.read_parquet(DATA_FILE, columns=["Date", "label"]).drop_nulls()
    .group_by("Date").agg(pl.col("label").std().alias("std"))
    .sort("Date")
)
REGIMES = ["calm", "normal", "turbulent"]
REGIME_COLORS = {"calm": "#cde2fb", "normal": "#6da7ec", "turbulent": "#184f95"}

per_day = per_day.sort("Date").with_columns(
    pl.col("std").shift(LAG).rolling_mean(SPAN).alias("trail_disp")   # dates T-26 ... T-7
)
td = per_day["trail_disp"].fill_null(np.nan).to_numpy()
pct = np.full(len(td), np.nan)
for i in range(len(td)):
    past = td[max(0, i - HISTORY):i]
    past = past[~np.isnan(past)]
    if len(past) >= MIN_HISTORY and not np.isnan(td[i]):
        pct[i] = (past < td[i]).mean()      # percentile vs the previous ~1 year
per_day = per_day.with_columns(pl.Series("disp_pct", pct).fill_nan(None)).with_columns(
    pl.when(pl.col("disp_pct").is_null()).then(None)
    .when(pl.col("disp_pct") < 1 / 3).then(pl.lit("calm"))
    .when(pl.col("disp_pct") < 2 / 3).then(pl.lit("normal"))
    .otherwise(pl.lit("turbulent")).alias("regime")
)
day_regime = per_day.select("Date", "regime")
daily = daily.drop("regime", strict=False).join(day_regime, on="Date", how="left")
preds = preds.drop("regime", strict=False).join(day_regime, on="Date", how="left")
test_days = (
    daily.group_by("Date").agg(pl.col("rank_ic").mean(), pl.col("regime").first())
    .sort("Date")
)
print("test days without a regime (too little history):", test_days["regime"].null_count())


def episodes(frame: pl.DataFrame, min_len: int = 5) -> pl.DataFrame:
    """Contiguous runs of the same regime over consecutive test days."""
    runs = (
        frame.with_columns((pl.col("regime") != pl.col("regime").shift()).fill_null(True)
                           .cum_sum().alias("run"))
        .group_by("run", maintain_order=True)
        .agg(pl.col("regime").first(), pl.col("Date").first().alias("start"),
             pl.col("Date").last().alias("end"), pl.len().alias("days"))
    )
    return runs.filter(pl.col("days") >= min_len)


rows = []
eps = episodes(test_days.drop_nulls("regime"))
for r in REGIMES:
    d = daily.filter(pl.col("regime") == r)
    ric = d.sort("Date")["rank_ic"].drop_nans().to_numpy()
    rows.append({
        "regime": r, "days": d["Date"].n_unique(),
        "episodes (>=5 days)": eps.filter(pl.col("regime") == r).height,
        "RankIC": ric.mean(), "RankIC 95% CI ±": nw_ci(ric),
        "IC": d["ic"].mean(), "RankIC>0 %": 100 * (ric > 0).mean(),
        "Q5-Q1": d["q5_minus_q1"].mean(),
    })
by_regime = pl.DataFrame(rows)
by_window = (
    daily.drop_nulls("regime").group_by(["window", "regime"])
    .agg(pl.len().alias("days"), pl.col("rank_ic").mean().alias("RankIC"))
    .sort(["window", "regime"])
)
with pl.Config(float_precision=4, tbl_rows=-1):
    display(by_regime)  # noqa: F821
    display(by_window.pivot(on="regime", index="window", values=["days", "RankIC"]))  # noqa: F821


In [ ]:
def shade(ax) -> None:
    """Shade the background by regime (all runs, including short ones)."""
    runs = episodes(test_days.drop_nulls("regime"), min_len=1)
    starts = runs["start"].to_list()
    ends = starts[1:] + [test_days["Date"].max()]
    for regime_name, s, e in zip(runs["regime"], starts, ends):
        ax.axvspan(s, e, color=REGIME_COLORS[regime_name], alpha=0.45, lw=0)


in_test = per_day.filter(pl.col("Date").is_between(test_days["Date"].min(), test_days["Date"].max()))
# Largest dispersion spikes in the test period, at least 60 days apart.
spikes = []
for row in in_test.sort("std", descending=True).iter_rows(named=True):
    if all(abs((row["Date"] - s).days) > 60 for s in spikes):
        spikes.append(row["Date"])
    if len(spikes) == 3:
        break

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 0.8, 1.2]})
dates = in_test["Date"].to_numpy()
axes[0].plot(dates, in_test["std"], color="#8f8e88", lw=0.8, label="daily dispersion (label std)")
axes[0].plot(dates, in_test["trail_disp"], color="#0b0b0b", lw=2, label="trailing measure (T-26 ... T-7)")
axes[0].set_title("1. Stock-specific dispersion: the trailing measure reacts only after a spike")
axes[1].plot(dates, in_test["disp_pct"], color="#0b0b0b", lw=1.5)
for level in (1 / 3, 2 / 3):
    axes[1].axhline(level, color="gray", ls="--", lw=1)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("percentile")
axes[1].set_title("2. Trailing dispersion vs its previous year: < 1/3 calm, > 2/3 turbulent")
t = test_days["Date"].to_numpy()
axes[2].plot(t, test_days["rank_ic"], color="#8f8e88", lw=0.6)
axes[2].plot(t, test_days["rank_ic"].rolling_mean(20, min_samples=5), color=WINDOW_COLORS[0],
             lw=2, label="Rank IC, 20-day mean")
axes[2].axhline(0, color="black", lw=0.8)
axes[2].axhline(test_days["rank_ic"].mean(), color="gray", ls="--", lw=1, label="overall mean")
axes[2].set_title("3. Daily Rank IC against the regime shading")
for ax in axes:
    shade(ax)
    for s in spikes:
        ax.axvline(s, color="#d03b3b", ls=":", lw=1.2)
for s in spikes:
    axes[0].annotate(f"spike {s}", (s, 1.0), xycoords=("data", "axes fraction"),
                     ha="left", va="top", fontsize=8, color="#d03b3b", xytext=(3, -3),
                     textcoords="offset points")
from matplotlib.patches import Patch  # noqa: E402
regime_handles = [Patch(color=REGIME_COLORS[r], alpha=0.45, label=r) for r in REGIMES]
axes[0].legend(loc="upper left", fontsize=8, bbox_to_anchor=(0, 0.9))
axes[2].legend(handles=axes[2].get_legend_handles_labels()[0] + regime_handles, loc="upper left",
               fontsize=8, ncol=5)
save(fig, "regime_timeline")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
xs = np.arange(len(REGIMES))
axes[0].bar(xs, by_regime["RankIC"], color=[REGIME_COLORS[r] for r in REGIMES], width=0.6,
            yerr=by_regime["RankIC 95% CI ±"], capsize=6, ecolor="#52514e")
axes[0].axhline(daily["rank_ic"].mean(), color="gray", ls="--", lw=1, label="all days")
for xi, row in zip(xs, by_regime.iter_rows(named=True)):
    axes[0].annotate(f"{row['days']} days\n{row['episodes (>=5 days)']} episodes", (xi, 0),
                     textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8,
                     color="#0b0b0b" if row["regime"] != "turbulent" else "white")
axes[0].set_xticks(xs, REGIMES)
axes[0].set_title("Mean daily Rank IC by regime (95% Newey–West CI)")
axes[0].legend(loc="upper left", fontsize=8)
for w in range(1, n_windows + 1):
    bw = by_window.filter((pl.col("window") == w) & (pl.col("days") >= 10))
    xw = [REGIMES.index(r) + (w - (n_windows + 1) / 2) * 0.06 for r in bw["regime"]]
    axes[1].plot(xw, bw["RankIC"], "-", color=WINDOW_COLORS[w - 1], lw=1, alpha=0.6)
    axes[1].scatter(xw, bw["RankIC"], s=bw["days"] * 3, color=WINDOW_COLORS[w - 1],
                    edgecolor="white", linewidth=1, label=f"W{w}", zorder=3)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(xs, REGIMES)
axes[1].set_title("Per window (dot size = days; regimes with < 10 days omitted)")
axes[1].legend(ncol=3, fontsize=8, loc="upper left")
save(fig, "regimes")


## 6. What changed vs the baseline: input columns only (raw + label history with max20 + feature deviations)

Baseline = Run-5 (48 raw features, purged 5-fold validation, retrain). Only the input columns differ.

| Category | Run 5 → run 10.1 | Why | Expected effect |
|---|---|---|---|
| Features | 48 raw → **149** = 48 raw + 5 `labelhist-max` + 96 `dev` | Give each stock its own past (realized labels, recent feature changes) | Small Rank IC gain; unknown effect on magnitudes |
| ↳ labelhist-max | `labelhist_last` (label of t−6), `labelhist_mean20`, `labelhist_mean60`, `labelhist_std20`, `labelhist_max20` (signed max over 20 dates) | Labels of dates ≤ t−6 are known on date t | Data prior: max20 alone has daily Rank IC −0.079 vs the label, but it is 0.85-correlated with std20; after controls only −0.004 |
| ↳ dev | `featureN_dev5`, `featureN_dev20` = feature − its mean over t−5…t−1 / t−20…t−1 | Is today's value unusual for this stock? | Alone (run 9): +0.0020 Rank IC, not significant |
| Rows / universe | Unchanged: identical train / validation / test rows in all 6 windows (only raw features + label must be finite) | History columns may be missing → LightGBM missing values | Paired comparison covers 100% of rows (2,028,817) |
| Missing history | `labelhist_mean60` missing for 16% of W1 history rows; < 0.4% elsewhere; dev ≈ 1.4% | Data start | W1 learns label history from fewer rows |
| Validation, params, schedule, seed | Unchanged (purged 5-fold, purge 10, same LightGBM params, seed 70563, 6 windows, final window skipped) | — | — |
| Code-only | Script imports `train_baseline.py` as a library; no intended effect | Shared protocol | None |

**Effective consequences.** Features ×3.1 (48 → 149). Trees per window 274 / 296 / 292 / 320 / 145 / 269 (mean 266) vs 374 / 512 / 408 / 563 / 286 / 371 (mean 419): −37%. Runtime 10.4 vs 11.1 min. Peak memory 3.4–3.8 vs 2.2 GB.

**Two feature groups changed at once.** Their effects are separated with runs 8.1 (raw + labelhist-max) and 9 (raw + dev), same protocol.


In [ ]:
assert COMPARE, "No baseline set: skip this section."
import difflib  # noqa: E402
import io  # noqa: E402
import tokenize  # noqa: E402


def code_without_comments(path: Path) -> list[str]:
    """Source lines with comments and blank lines removed (NOTE comments aren't setup changes)."""
    out, last_line = [], -1
    tokens = tokenize.generate_tokens(io.StringIO(path.read_text(encoding="utf-8")).readline)
    kept = [t for t in tokens if t.type != tokenize.COMMENT]
    src = tokenize.untokenize(kept)
    for line in src.splitlines():
        if line.strip():
            out.append(line.rstrip())
    return out


def archived_script(run_dir: Path) -> Path | None:
    """The training script copy archived in the run folder (largest .py, if several)."""
    scripts = [p for p in run_dir.glob("*.py")]
    return max(scripts, key=lambda p: p.stat().st_size) if scripts else None


run_script, base_script = archived_script(RUN_DIR), archived_script(BASELINE_DIR)
print(f"run script:      {run_script.name if run_script else 'NOT ARCHIVED'}")
print(f"baseline script: {base_script.name if base_script else 'NOT ARCHIVED'}")
if run_script and base_script:
    diff = list(difflib.unified_diff(code_without_comments(base_script), code_without_comments(run_script),
                                     fromfile=f"baseline/{base_script.name}", tofile=f"run/{run_script.name}",
                                     n=1, lineterm=""))
    print(f"\n--- code diff (comments and blank lines ignored): {sum(l.startswith(('+', '-')) and not l.startswith(('+++', '---')) for l in diff)} changed lines ---")
    print("\n".join(diff) if diff else "(no code differences)")

# Per-window data and selection facts, matched by test start date.
b_by_start = {r["test_start"]: r for r in b_results}
rows = []
for k, r in enumerate(results):
    b = b_by_start.get(r["test_start"], {})
    rows.append({"window": k + 1, "test_start": r["test_start"]}
                | {f"{key} base": b.get(key) for key in ("train_rows", "validation_rows", "test_rows", "features")}
                | {f"{key} run": r.get(key) for key in ("train_rows", "validation_rows", "test_rows", "features")})
setup_facts = pl.DataFrame(rows)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=250):
    display(setup_facts)  # noqa: F821


def selected_features(run_dir: Path) -> dict[str, set[str]]:
    """Selected features per window (keyed by test start), from models/*.features.json."""
    out = {}
    for f in (run_dir / "models").glob("*.features.json"):
        m = re.search(r"_test_(\d{4}-\d{2}-\d{2})_", f.name)
        if m:
            out[m[1]] = set(json.loads(f.read_text())["features"])
    return out


run_feats, base_feats = selected_features(RUN_DIR), selected_features(BASELINE_DIR)
for k, r in enumerate(results):
    a, b = run_feats.get(r["test_start"]), base_feats.get(r["test_start"])
    if a is None or b is None:
        print(f"W{k + 1}: feature lists not available")
        continue
    print(f"W{k + 1}: {len(b)} -> {len(a)} features; added {sorted(a - b) or '-'}; removed {sorted(b - a) or '-'}")


## 7. Comparison with the baseline: `run-5_baseline-all-feat-purged-5-fold-val-retrain`

**Is the run better than the baseline, and where?** Market conditions move
every model's IC far more than model changes do: whole windows rise and fall
together. So the two models are compared **paired**: on the **same (Date,
Code) rows**, day by day.

- **Coverage:** only rows present in both runs are compared. If the run
  changes the universe (e.g. drops new listings), the comparison is
  restricted to the common rows; the run's own section 1–5 metrics still use
  all of its rows.
- **Daily delta** = run − baseline for Rank IC, IC and Q5−Q1 on each date.
  Its mean is the improvement, with a **Newey–West 95% CI** (deltas are
  autocorrelated too). **% days better** = share of dates where the run's
  Rank IC is higher.
- **Where:** the delta per window, per month and per ex-ante regime. A real
  improvement should be positive in **most windows**, not carried by one.
- **Agreement:** the daily rank correlation between the two models'
  predictions. Near 1 means the run barely changed what the model believes.
- **Magnitudes** (as important as ranking: the predictions feed a portfolio
  optimizer): calibration slope, R² and RMSE per window on the common rows,
  and quintile profiles, for both models.

Rules of thumb: an improvement of ~+0.005 Rank IC is meaningful **if** its CI
excludes 0 and it holds in most windows. Never choose a variant using the
final reserved window.


In [ ]:
assert COMPARE, "No baseline set: skip this section."

# ---- Align the two runs on common (Date, Code) rows -----------------------------
pair = (
    preds.select("window", "Date", "Code", "label", pl.col("Prediction").alias("p_run"))
    .join(b_preds.select("Date", "Code", pl.col("label").alias("label_base"),
                         pl.col("Prediction").alias("p_base")),
          on=["Date", "Code"], how="inner")
)
label_mismatch = pair.filter((pl.col("label") - pl.col("label_base")).abs() > 1e-6).height
print(f"rows: run {preds.height:,} | baseline {b_preds.height:,} | common {pair.height:,} "
      f"({100 * pair.height / preds.height:.1f}% of run) on {pair['Date'].n_unique()} dates; "
      f"label mismatches: {label_mismatch}")
assert pair.height > 0, "no common rows"
if label_mismatch:
    print("  WARNING: labels differ between runs on some rows (transformed labels?). "
          "The comparison uses the run's labels.")

# ---- Paired daily metrics ------------------------------------------------------------
quint_expr = [
    ((pl.col(p).rank("ordinal").over("Date").cast(pl.Int64) - 1) * 5 // pl.len().over("Date") + 1)
    .alias(f"q_{p[2:]}") for p in ("p_run", "p_base")
]
pair = pair.with_columns(quint_expr)


def q_mean(q_col: str, q: int) -> pl.Expr:
    return pl.col("label").filter(pl.col(q_col) == q).mean()


paired = (
    pair.group_by(["window", "Date"])
    .agg(
        pl.corr("p_run", "label", method="spearman").alias("ric_run"),
        pl.corr("p_base", "label", method="spearman").alias("ric_base"),
        pl.corr("p_run", "label").alias("ic_run"),
        pl.corr("p_base", "label").alias("ic_base"),
        (q_mean("q_run", 5) - q_mean("q_run", 1)).alias("spread_run"),
        (q_mean("q_base", 5) - q_mean("q_base", 1)).alias("spread_base"),
        q_mean("q_run", 1).alias("q1_run"), q_mean("q_base", 1).alias("q1_base"),
        q_mean("q_run", 5).alias("q5_run"), q_mean("q_base", 5).alias("q5_base"),
        pl.corr("p_run", "p_base", method="spearman").alias("pred_agreement"),
    )
    .with_columns(
        (pl.col("ric_run") - pl.col("ric_base")).alias("d_ric"),
        (pl.col("ic_run") - pl.col("ic_base")).alias("d_ic"),
        (pl.col("spread_run") - pl.col("spread_base")).alias("d_spread"),
        pl.col("Date").dt.strftime("%Y-%m").alias("month"),
    )
    .join(day_regime, on="Date", how="left")
    .sort("Date")
)


def delta_row(d: pl.DataFrame, **label) -> dict:
    """Summary of paired daily deltas for one slice of dates."""
    x = d.sort("Date")["d_ric"].drop_nans().to_numpy()
    return {
        **label,
        "days": d.height,
        "RankIC base": d["ric_base"].mean(),
        "RankIC run": d["ric_run"].mean(),
        "dRankIC": x.mean(),
        "95% CI ±": nw_ci(x) if len(x) > 10 else float("nan"),
        "t(NW)": newey_west_t(x) if len(x) > 10 else float("nan"),
        "run better %": 100 * (x > 0).mean(),
        "dIC": d["d_ic"].mean(),
        "dQ5-Q1": d["d_spread"].mean(),
        "Q1 base": d["q1_base"].mean(), "Q1 run": d["q1_run"].mean(),
        "Q5 base": d["q5_base"].mean(), "Q5 run": d["q5_run"].mean(),
        "agreement": d["pred_agreement"].mean(),
    }


overall = delta_row(paired, scope="all")
by_window_delta = pl.DataFrame(
    [delta_row(paired.filter(pl.col("window") == w), scope=f"W{w}")
     for w in sorted(paired["window"].unique())]
)
by_month_delta = (
    paired.group_by("month").agg(
        pl.len().alias("days"), pl.col("d_ric").mean().alias("dRankIC"),
        pl.col("ric_base").mean().alias("RankIC base"), pl.col("ric_run").mean().alias("RankIC run"),
        (100 * (pl.col("d_ric") > 0).mean()).alias("run better %"),
    ).sort("month")
)
by_regime_delta = pl.DataFrame(
    [delta_row(paired.filter(pl.col("regime") == r), scope=r) for r in REGIMES
     if paired.filter(pl.col("regime") == r).height]
)

# ---- Calibration and per-window facts for both models ---------------------------
def slope(frame: pl.DataFrame, p: str) -> pl.Expr:
    return (pl.cov(p, "label") / pl.col(p).var()).alias(f"slope_{p[2:]}")


def r2(p: str) -> pl.Expr:
    return (1 - ((pl.col("label") - pl.col(p)) ** 2).sum()
            / ((pl.col("label") - pl.col("label").mean()) ** 2).sum()).alias(f"R2_{p[2:]}")


def rmse(p: str) -> pl.Expr:
    return ((pl.col("label") - pl.col(p)) ** 2).mean().sqrt().alias(f"RMSE_{p[2:]}")


magnitude_exprs = [slope(pair, "p_base"), slope(pair, "p_run"), r2("p_base"), r2("p_run"),
                   rmse("p_base"), rmse("p_run")]
calib = pair.group_by("window").agg(magnitude_exprs).sort("window")
magnitude_all = pair.select(magnitude_exprs)
b_by_start = {r["test_start"]: r for r in b_results}
facts = pl.DataFrame([{
    "window": k + 1,
    "best_iter base": b_by_start.get(r["test_start"], {}).get("best_iteration"),
    "best_iter run": r.get("best_iteration"),
    "features base": b_by_start.get(r["test_start"], {}).get("features"),
    "features run": r.get("features"),
    "test R2 base": b_by_start.get(r["test_start"], {}).get("r2"),
    "test R2 run": r.get("r2"),
    "runtime base": b_by_start.get(r["test_start"], {}).get("runtime"),
    "runtime run": r.get("runtime"),
} for k, r in enumerate(results)]).join(calib, on="window", how="left")

print(f"\nOVERALL: Rank IC {overall['RankIC base']:.4f} -> {overall['RankIC run']:.4f}  "
      f"dRankIC {overall['dRankIC']:+.4f} ± {overall['95% CI ±']:.4f} (t NW {overall['t(NW)']:.1f}), "
      f"run better on {overall['run better %']:.0f}% of days; dIC {overall['dIC']:+.4f}; "
      f"dQ5-Q1 {overall['dQ5-Q1']:+.4f}; prediction agreement {overall['agreement']:.2f}")
m = magnitude_all.row(0, named=True)
print(f"MAGNITUDES (common rows): calibration slope {m['slope_base']:.3f} -> {m['slope_run']:.3f}; "
      f"R2 {m['R2_base']:.4f} -> {m['R2_run']:.4f}; RMSE {m['RMSE_base']:.5f} -> {m['RMSE_run']:.5f}")
n_pos = (by_window_delta["dRankIC"] > 0).sum()
print(f"windows with dRankIC > 0: {n_pos} of {by_window_delta.height}")
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    display(by_window_delta)  # noqa: F821
    display(by_regime_delta)  # noqa: F821
    display(by_month_delta)  # noqa: F821
    display(facts)  # noqa: F821


In [ ]:
assert COMPARE, "No baseline set: skip this section."
run_name, base_name = RUN_DIR.name, BASELINE_DIR.name
win_starts = [paired.filter(pl.col("window") == w)["Date"].min() for w in sorted(paired["window"].unique())]

# ---- 1. Where in time does the run gain or lose? ------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
t = paired["Date"].to_numpy()
axes[0].plot(t, paired["d_ric"].fill_nan(0).cum_sum(), color=MODEL_COLORS["run"], label="cumulative ΔRank IC")
axes[0].plot(t, paired["d_ic"].fill_nan(0).cum_sum(), color="#eb6834", lw=1.5, label="cumulative ΔPearson IC")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title(f"Cumulative daily difference, {run_name} − {base_name} (rising = run ahead)")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].plot(t, paired["d_ric"], color="#b5b4ad", lw=0.6)
axes[1].plot(t, paired["d_ric"].rolling_mean(20, min_samples=5), color=MODEL_COLORS["run"],
             label="ΔRank IC, 20-day mean")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Daily ΔRank IC against the ex-ante regime shading")
for ax in axes:
    shade(ax)
    for s in win_starts[1:]:
        ax.axvline(s, color="gray", ls="--", lw=0.8)
axes[1].legend(handles=axes[1].get_legend_handles_labels()[0]
               + [Patch(color=REGIME_COLORS[r], alpha=0.45, label=r) for r in REGIMES],
               loc="upper left", fontsize=8, ncol=4)
save(fig, "compare_timeline")

# ---- 2. Is the gain consistent across windows, regimes, months? ---------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1, 0.7, 1.6]})
wl = by_window_delta["scope"].to_list()
wcol = [WINDOW_COLORS[int(s[1:]) - 1] for s in wl]
axes[0].bar(wl, by_window_delta["dRankIC"], color=wcol, width=0.6,
            yerr=by_window_delta["95% CI ±"], capsize=5, ecolor="#52514e")
axes[0].set_title("ΔRank IC per window (95% NW CI)")
rl = by_regime_delta["scope"].to_list()
axes[1].bar(rl, by_regime_delta["dRankIC"], color=[REGIME_COLORS[r] for r in rl], width=0.6,
            yerr=by_regime_delta["95% CI ±"], capsize=5, ecolor="#52514e")
axes[1].set_title("ΔRank IC per ex-ante regime")
month_window = paired.group_by("month").agg(pl.col("window").mode().first()).sort("month")
mcol = [WINDOW_COLORS[w - 1] for w in month_window["window"]]
xm = np.arange(by_month_delta.height)
axes[2].bar(xm, by_month_delta["dRankIC"], color=mcol, width=0.8)
axes[2].set_xticks(xm, by_month_delta["month"], rotation=70, fontsize=7)
axes[2].set_title("ΔRank IC per month (colour = window)")
for ax in axes:
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(overall["dRankIC"], color=MODEL_COLORS["run"], ls="--", lw=1)
save(fig, "compare_breakdown")

# ---- 3. Levels and magnitudes for both models ----------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
xw = np.arange(len(wl))
axes[0].plot(xw, by_window_delta["RankIC base"], "o-", color=MODEL_COLORS["baseline"], label=base_name, ms=7)
axes[0].plot(xw, by_window_delta["RankIC run"], "o-", color=MODEL_COLORS["run"], label=run_name, ms=7)
axes[0].set_xticks(xw, wl)
axes[0].set_title("Mean daily Rank IC per window")
qcols = [f"Q{q}" for q in (1, 5)]
qb = [overall["Q1 base"], overall["Q5 base"]]
qr = [overall["Q1 run"], overall["Q5 run"]]
xq = np.arange(2)
axes[1].bar(xq - 0.18, qb, width=0.35, color=MODEL_COLORS["baseline"], label=base_name)
axes[1].bar(xq + 0.18, qr, width=0.35, color=MODEL_COLORS["run"], label=run_name)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(xq, ["bottom quintile (Q1)", "top quintile (Q5)"])
axes[1].set_title("Mean label of the extreme quintiles")
axes[2].plot(xw, facts["slope_base"], "o-", color=MODEL_COLORS["baseline"], label=base_name, ms=7)
axes[2].plot(xw, facts["slope_run"], "o-", color=MODEL_COLORS["run"], label=run_name, ms=7)
axes[2].axhline(1, color="gray", ls="--", lw=1)
axes[2].set_xticks(xw, [f"W{w}" for w in facts["window"]])
axes[2].set_title("Calibration slope per window (1 = calibrated)")
for ax in axes:
    ax.legend(fontsize=8)
save(fig, "compare_levels")
print(f"Mean daily prediction agreement (rank corr run vs baseline): {overall['agreement']:.3f}")


## Findings

Numbers: test, 6 windows, 720 dates, all rows paired with run 5 and run 2. CIs and t are Newey–West with 10 lags (from the shared brief). The notebook's cell 10 uses 4 lags: ±0.0067, t 2.5 for the same Δ. "dm" = after per-day demeaning (the reference is Run-5 demeaned).

**This run on its own**

1. **Ranking.** Rank IC 0.0717 (NW t 15.0), Pearson IC 0.0956, Kendall 0.0484, ICIR 1.20, Pearson IC > 0 on 89% of days. Highest Rank IC and Kendall of all 8 runs on the leaderboard. Pearson / Rank = 1.33 (run 5: 1.56): less driven by tail stocks.
2. **Consistency.** Window Rank IC 0.0599–0.0966. Medians ≈ means (W5: median 0.0853, mean 0.0841). Rank IC > 0 on 79–90% of days per window. The worst month is Jan-2024 (−0.036); run 5 is also negative then (−0.028), so it is the period.
3. **Quintiles.** The edge is in Q1: mean label Q1 −0.0056, Q2–Q5 +0.0011 … +0.0023. Q5−Q1 0.0079. W6 has a flat top (Q4 0.0001, Q5 0.0003).
4. **Calibration.** Demeaned slope per window 1.21 / 1.28 / 1.16 / **0.65** / 0.97 / 1.07; pooled 0.978. Months below 0.5: Sep-2023 0.20, Feb-2024 0.43, Sep-2024 0.41, Oct-2024 0.04, May-2025 0.29.
5. **Val vs test R².** Test ≈ or > val in every window except W4 (val 0.0078, test 0.0033). W4 contains Oct-2024 (R² −0.015, slope 0.04) after the Sep-2024 dispersion spike, and run 5 is also weak there (0.0050). So the test period, not a leak. W4 also has the biggest train–val gap (0.042 vs 0.0078).
6. **Regimes (ex-ante).** Calm 0.0536 ± 0.0152 (8 episodes), normal 0.0690 ± 0.0136 (11), turbulent 0.1011 ± 0.0163 (3 episodes: a hint only). Same order as run 5.

**Comparison with run 5**

7. **Overall.** ΔRank IC **+0.0085 ± 0.0073 (t 2.3)**, run better on 56% of days, 5 of 6 windows (W2 −0.0056). ΔPearson IC −0.0028. ΔQ5−Q1 −0.0006 (0.0079 vs 0.0085).
8. **Concentration over time.** W1 and W5 give 81% of the summed daily gain (29% + 52%); W2 −11%. A post-hoc split (cut points read off the cumulative plot) shows a full flat year:

   | period | days | ΔRank IC (t) |
   |---|---|---|
   | Dec-2022 – Jul-2023 | 141 | +0.0182 (2.2) |
   | Aug-2023 – Aug-2024 | 265 | −0.0048 (−0.8) |
   | Sep-2024 – Jun-2025 | 197 | +0.0217 (4.3) |
   | Jul-2025 – Dec-2025 | 117 | +0.0048 (0.5) |

   Months: 22 of 37 positive.
9. **W5 gain is spread, not a few months.** 6 of 7 months positive; only May-2025 is negative (−0.0114). Dropping any one month leaves +0.0224 … +0.0340. Median day +0.0266, 10%-trimmed mean +0.0248, NW t 3.4. Most of it is label history: 8.1 − 5 is +0.0226 in W5; dev adds +0.0042.
10. **Regimes.** Δ calm +0.0064 (t 1.2), normal +0.0095 (t 1.7), turbulent +0.0105 (t 2.0, 3 episodes). Not carried by one regime; none is significant alone.
11. **Quintiles: better order, weaker extremes.** In rank terms both ends improve. Rank IC contribution of the run's own Q1: 0.0526 → 0.0562 (+0.0036); Q5: 0.0086 → 0.0117 (+0.0031). In W5, Q5 leads (+0.0139 vs Q1 +0.0073; Q5 mean label 0.0020 → 0.0030). In label units the extremes weaken: Q1 −0.0061 → −0.0056, Q5 0.0024 → 0.0023. The run orders more stocks correctly but catches fewer of the largest losers. This matches ΔPearson −0.0028 and the cumulative ΔPearson drifting to about −2 by Dec-2025.
12. **Magnitudes are worse.** R² dm 0.00942 → 0.00872 (−7%); worse in W1, W4, W6 (W4 0.00655 → 0.00389), better in W2, W3, W5 (W5 0.00443 → 0.00545). RMSE dm 0.055818 → 0.055838. Mean per-window |slope − 1| 0.178 → 0.185 (W4 0.75 → 0.65). The pooled slope is closer to 1 (0.954 → 0.978), so the R² loss comes from the lower Pearson IC, not from scale. Rescaling won't recover it.
13. **Agreement.** Mean daily rank agreement 0.63 (W6 0.49); pooled correlation 0.82. A clearly different signal from run 5.
14. **Dev on top of label history (10.1 − 8.1).** +0.0019 ± 0.0018 (t 2.1), 4 of 6 windows, 24 of 37 months, ΔR² dm +0.00013. Its pattern repeats dev alone (9 − 5): per-window correlation 0.85, daily 0.61. Like run 9, it pays from Sep-2024 on (+0.0069, t 4.1; Jul–Dec-2025 +0.0035, t 4.3) and is ≈ 0 before. The effects add up: (8.1 − 5) + (9 − 5) = +0.0087 ≈ +0.0085.
15. **Cost.** Runtime 10.4 vs 11.1 min; peak memory 3.4–3.8 vs 2.2 GB; 37% fewer trees (W5: 145).
16. **Validation vs test.** Mean val ΔRank IC +0.0060 vs test +0.0085: right sign and size. But per window the correlation is −0.66 (4/6 signs agree). W5 has val −0.0005, test +0.0268. R² dm: correlation +0.01, 2/6 signs. Validation can pick the family but cannot say where it pays.
17. **Selection.** 12 configurations have been compared on these windows; 10.1 is the best of 6 run together. With a Bonferroni correction over the 6, t 2.3 would need ≈ 2.6. The true gain is likely positive but probably below +0.0085.

**Comparison with run 2**

18. ΔRank IC **+0.0136 ± 0.0070 (t 3.8)**, 6 of 6 windows. ΔPearson +0.0012. R² dm 0.00872 vs 0.00831. Pooled dm slope 0.978 vs 0.848. Q5−Q1 0.0079 vs 0.0080. Mean per-window |slope − 1| 0.185 vs 0.160 (slightly worse).

**Verdicts**

19. **vs run 5: not better by the strict criteria.** It is a mixed result: a significant ranking gain (CI excludes 0, 5/6 windows) bought with worse magnitudes (R² dm −7%, Pearson −0.0028, Q5−Q1 −0.0006). The gain is concentrated: 81% in two windows and none for a year. Not "worse" either.
20. **vs run 2: better.** A ranking-led gain, with magnitudes not worse (R² and slope better; per-window slope spread slightly worse).


## Summary: run 10.1 (raw + labelhist-max + dev) vs Run-5 (demeaned), and vs run 2

**Setup.** Only the inputs changed: 48 raw → 149 columns (5 label-history columns with max20, 96 feature deviations). Same rows, protocol, params and seed. 6 windows, 720 test dates (2022-12-29 → 2025-12-18). Final window held back.

**Overall performance.** Ranking: Rank IC 0.0717 (Pearson IC 0.0956), ICIR 1.20, Pearson IC > 0 on 89% of days, NW t 15.0, Q5−Q1 0.0079. Best Rank IC of all runs. Magnitude: R² dm 0.00872, pooled dm slope 0.978, per-window slopes 0.65–1.28, RMSE dm 0.055838. Cost: 10.4 min, 3.8 GB peak, 37% fewer trees.

**Comparison with run 5.** ΔRank IC +0.0085 ± 0.0073 (t 2.3). Run better on 56% of days and in 5 of 6 windows. W1 (+0.0146) and W5 (+0.0268) give 81% of the gain; Aug-2023 – Aug-2024 is flat (−0.0048, t −0.8). All regimes are positive (+0.0064 … +0.0105). Both quintile ends rank better, but the extreme losers are caught less (Q1 −0.0061 → −0.0056). ΔPearson −0.0028, ΔQ5−Q1 −0.0006, R² dm −7%. Daily agreement 0.63. **Verdict: not better by the strict criteria.** Ranking is better; magnitudes are worse.

**Comparison with run 2.** ΔRank IC +0.0136 ± 0.0070 (t 3.8), 6/6 windows. ΔPearson +0.0012; R² dm 0.00872 vs 0.00831. **Verdict: better** (ranking-led; magnitudes not worse).

**Where it excels.**
- W5: +0.0268 (t 3.4), spread over 6 of 7 months; dropping any month leaves ≥ +0.0224.
- Sep-2024 – Jun-2025: +0.0217 (t 4.3). Both feature groups add there.
- Rank-based ordering at both ends: Q1 contribution +0.0036, Q5 +0.0031.

**Where it stumbles.**
- Magnitudes: R² dm lower in W1, W4, W6 (W4 0.00655 → 0.00389); W4 slope 0.65.
- Tail stocks: lower Pearson IC and Q5−Q1; cumulative ΔPearson ≈ −2 by Dec-2025.
- No gain for a whole year (Aug-2023 – Aug-2024).

**Probable reasons.**
1. Label history (model effect): 8.1 − 5 gives +0.0067 of the +0.0085 and the same W1 / W5 bumps, plus the same R² loss.
2. Feature deviations (model effect): +0.0019 (t 2.1) on top, only from Sep-2024 on, as in run 9.
3. Period: the flat year and the W1 / W5 bumps are shared by 8.1 and partly by 9. The test period sets where the gain shows.

**What this run achieved and what to try next.**
- The best ranking so far, still with a magnitude cost. Validation got the average right (+0.0060) but not the windows (corr −0.66).
- Try raw + labelhist + lags: lags (run 11) gave the best magnitudes (R² dm 0.00997).
- Ablate label history (std20 only vs max20 only) to find which column drives the ranking gain and the R² loss.
- Blend 10.1 with run 5 or 11, weight fitted on validation, to recover Pearson IC and R².

**Hypotheses in brief.** Helped most: label history (max20 / std20) sorting stocks by recent extremes. Helped a little: feature deviations, from late 2024. Hurt: magnitude of the extreme losers (lower Pearson, R²). Mattered little: cost. The test period moves the gain more than either feature group.


## Hypotheses: what drove the difference vs Run-5 (demeaned)

**Likely improvements.**
1. **Label history ranks stocks by their recent extremes.** Change: `labelhist_max20`, `std20`, means and last label. Mechanism: a stock's recent label spread / best label marks lottery-like stocks; the trees use it to order the middle of the distribution better. Evidence: 8.1 − 5 is +0.0067 (t 1.8) and has the same W1 (+0.0156) and W5 (+0.0226) bumps as 10.1. Rank-IC contribution improves at both ends (Q1 +0.0036, Q5 +0.0031). In W5 Q5 leads (+0.0139). Confidence: medium. The average is positive on both validation (+0.0075 for 8.1) and test, but the window pattern is not stable (val–test corr −0.72).
2. **Feature deviations add a small, separate gain.** Mechanism: "is today unusual for this stock" helps after late 2024. Evidence: 10.1 − 8.1 +0.0019 (t 2.1); same window pattern as 9 − 5 (corr 0.85); gain only in Sep-2024 onward (+0.0069, t 4.1; then +0.0035, t 4.3). Effects add: +0.0067 + 0.0020 ≈ +0.0085. Confidence: medium-high.

**Likely degradations.**
1. **Weaker magnitudes at the tails.** Mechanism: label-history columns pull the ordering toward "moderately" bad stocks and away from the most extreme losers. Evidence: Q1 mean label −0.0061 → −0.0056, ΔPearson −0.0028, R² dm −7%. 8.1 shows the same R² loss (0.00858). Pooled slope 0.978 is close to 1, so it is not a scale error. Confidence: medium.
2. **Fewer trees.** Mean 266 vs 419 (−37%); W5 only 145. Early stopping stops sooner with more columns, which may leave less magnitude fitted. Confidence: low (W5 has the fewest trees and the best gain).
3. **Cost.** Memory +70% (3.8 vs 2.2 GB); runtime −6%. Minor.

**What matters most / what matters little.** Label history > period > deviations > cost. Max vs min version: 10.1 − 10.2 +0.0026 (t 3.1), a small edge for max20. The test period matters more than either group: the Aug-2023 – Aug-2024 flat year is shared by 8.1 (−0.0035) and 9 (−0.0032).

**Confounding.** Two groups changed at once. Runs 8.1 and 9 separate them, and the effects add, so the confounding is small. Within label history, max20 and std20 are 0.85-correlated and cannot be separated by these runs. They act on the same quintiles.

**Experiments to test them.**
1. **Label-history ablation:** raw + {std20 only} vs raw + {max20 only} vs raw + {means + last only}. Confirms H-improvement-1 if one column carries most of the +0.0067 and the R² loss.
2. **raw + labelhist-max + lags:** tests whether lags (best R² so far, run 11: 0.00997) keep the ranking gain while fixing magnitudes. Confirmed if Rank IC ≥ 0.070 and R² dm ≥ 0.0094.
3. **Blend 10.1 with run 5 (or 11)**, weight fitted on validation only. If Rank IC stays ≥ 0.070 with R² dm ≥ run 5, the degradation is a separable tail effect.
4. **Same setup, 2 more seeds** for 10.1 and 8.1: checks that the W1 / W5 concentration is not seed noise (seed ceiling so far ±0.0004).


## Leaderboard: runs side by side

Each run's **own** test-set metrics on the same 720 test dates. In each
column (first table) or row (other tables), the best value is **bold**.

- **Rank IC, Pearson IC, Kendall:** means of the daily values over all test
  dates. Kendall is daily, unlike the training script's pooled `kt` (I-4).
- **R²:** pooled over all test rows (first table) or over each window's rows.
- **ICIR:** mean(daily Pearson IC) / std, per window, averaged over the
  windows (unscaled, comparable across windows).

Read with care:
- These are unpaired levels. The paired tests (section 7) are what decide
  significance: a gap of ~0.001 Rank IC is within noise.
- Seed noise is not measured yet.
- Every extra variant compared on the same windows inflates the best one's
  apparent edge.


In [ ]:
from IPython.display import Markdown, display

LEADERBOARD_DIRS = ["/workspaces/TimeSeries48/runs/run-2_original-baseline-six-windows", "/workspaces/TimeSeries48/runs/run-5_baseline-all-feat-purged-5-fold-val-retrain", "/workspaces/TimeSeries48/runs/run-10.1_multi-day-rawfeat+featdeviation+labelhist-max-p5fcv", "/workspaces/TimeSeries48/runs/run-10.2_multi-day-rawfeat+featdeviation+labelhist-min-p5fcv", "/workspaces/TimeSeries48/runs/run-11_multi-day-rawfeat+lag1feat+lag5feat-p5fcv", "/workspaces/TimeSeries48/runs/run-8.1_multi-day-rawfeat+labelhist-max-p5fcv", "/workspaces/TimeSeries48/runs/run-8.2_multi-day-rawfeat+labelhist-min-p5fcv", "/workspaces/TimeSeries48/runs/run-9_multi-day-rawfeat+featdeviation-p5fcv"]  # Run folders to compare (set by the skill).


def short_name(path: Path) -> str:
    """'run-5_baseline-all-feat-...' -> 'run-5'."""
    return path.name.split("_")[0]


def r2_expr() -> pl.Expr:
    """Pooled R² of Prediction for label."""
    resid = ((pl.col("label") - pl.col("Prediction")) ** 2).sum()
    total = ((pl.col("label") - pl.col("label").mean()) ** 2).sum()
    return (1 - resid / total).alias("r2")


def run_summary(run_dir: Path) -> tuple[dict[str, float], pl.DataFrame]:
    """Headline test metrics over all dates, and per-window Rank IC / R² / ICIR."""
    _, p, _ = load_run(run_dir)
    d, _, _ = daily_metrics(p)
    per_window = (
        p.group_by("window").agg(r2_expr())
        .join(d.group_by("window").agg(
            pl.col("rank_ic").mean(),
            (pl.col("ic").mean() / pl.col("ic").std()).alias("icir"),
        ), on="window")
        .sort("window")
    )
    head = {
        "Rank IC": d["rank_ic"].mean(),
        "Pearson IC": d["ic"].mean(),
        "Kendall": d["kendall"].mean(),
        "R² (all rows)": p.select(r2_expr()).item(),
        "ICIR (window mean)": per_window["icir"].mean(),
    }
    return head, per_window


def bold_table(row_names: list[str], col_names: list[str], values: np.ndarray,
               fmts: list[str], best_per: str) -> str:
    """Markdown table; the highest shown value per 'row' or 'column' is bold."""
    shown = [[format(values[i, j], fmts[j if best_per == "column" else 0])
              for j in range(len(col_names))] for i in range(len(row_names))]
    num = np.array([[float(s) for s in row] for row in shown])
    best = num.max(axis=0 if best_per == "column" else 1)
    lines = ["| | " + " | ".join(col_names) + " |",
             "|---|" + "---|" * len(col_names)]
    for i, name in enumerate(row_names):
        cells = [f"**{s}**" if num[i, j] == (best[j] if best_per == "column" else best[i])
                 else s for j, s in enumerate(shown[i])]
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


dirs = [Path(p) for p in LEADERBOARD_DIRS]
if RUN_DIR.resolve() not in [p.resolve() for p in dirs]:
    dirs.append(RUN_DIR)
names = [short_name(p) for p in dirs]
summaries = {name: run_summary(p) for name, p in zip(names, dirs, strict=True)}

metrics = list(next(iter(summaries.values()))[0])
head_values = np.array([[summaries[n][0][m] for m in metrics] for n in names])
head_md = bold_table(names, metrics, head_values,
                     ["0.4f", "0.4f", "0.4f", "0.5f", "0.3f"], best_per="column")

window_rows = [f"W{k + 1} ({r['test_start']} → {r['test_end']})"
               for k, r in enumerate(results)] + ["all test dates"]
rank_ic = np.array([[*summaries[n][1]["rank_ic"].to_list(), summaries[n][0]["Rank IC"]]
                    for n in names]).T
r2 = np.array([[*summaries[n][1]["r2"].to_list(), summaries[n][0]["R² (all rows)"]]
               for n in names]).T
rank_md = bold_table(window_rows, names, rank_ic, ["0.4f"], best_per="row")
r2_md = bold_table(window_rows, names, r2, ["0.5f"], best_per="row")

legend = "\n".join(f"- **{n}**: `{p.name}`" for n, p in zip(names, dirs, strict=True))
text = (f"**Runs**\n\n{legend}\n\n**Headline test metrics** (best per column in bold)\n\n"
        f"{head_md}\n\n**Rank IC per window** (best per row in bold)\n\n{rank_md}\n\n"
        f"**R² per window** (best per row in bold)\n\n{r2_md}")
display(Markdown(text))
print(text)  # Plain text too, for headless runs.
